In [1]:
import sys; sys.path.append("../src")
import numpy as np, tensorflow as tf
from tensorflow.keras import layers
import pipeline

tf.keras.utils.set_random_seed(42)

# same pipeline as before, with one-hot labels so label smoothing can be applied
to_onehot = lambda x, y: (x, tf.one_hot(y, 8))
train_ds = pipeline.make_dataset(pipeline.train_df, shuffle=True, augment_data=True).map(to_onehot)
val_ds = pipeline.make_dataset(pipeline.val_df, shuffle=False, augment_data=False).map(to_onehot)

model = tf.keras.models.load_model("../models/stage1_best.keras")   # EfficientNetB0 after stage 1
base = model.get_layer("efficientnetb0")
base.trainable = True
for layer in base.layers:
    layer.trainable = (layer.name.startswith(("block6", "block7", "top"))
                       and not isinstance(layer, layers.BatchNormalization))

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=0.05),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)
print("Trainable params:", sum(int(np.prod(w.shape)) for w in model.trainable_weights))

I0000 00:00:1791785975.983487    1875 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791785976.023778    1875 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791785977.183704    1875 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791785978.754508    1875 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6994 MB mem

Trainable params: 3140132


In [2]:
import os
os.makedirs("../models", exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint("../models/reg_stage2_best.keras",
                                       monitor="val_accuracy", save_best_only=True, verbose=1),
    tf.keras.callbacks.CSVLogger("../models/reg_stage2_log.csv"),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3,
                                         patience=1, min_lr=1e-6, verbose=1),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=callbacks)
model.save("../models/reg_final.keras")   # the weights restored by early stopping

Epoch 1/12


/home/suemon/enge707-kaggle/venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1791786062.564674    2001 service.cc:153] XLA service 0x7698600850f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1791786062.564702    2001 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3080, Compute Capability 8.6 (Driver: 13.4.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1791786063.066561    2001 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1791786064.442688    2001 cuda_dnn.cc:461] Loaded cuDNN version 92700
I0000 00:00:1791786064.693163    2001 dot_merger.cc:481] Merg

1591/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9569 - loss: 0.6265

I0000 00:00:1791786114.582682    2001 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_21191__.254
E0000 00:00:1791786116.070037    2001 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1791786118.623557    2001 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


1592/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9570 - loss: 0.6265

E0000 00:00:1791786137.471585    1999 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.



Epoch 1: val_accuracy improved from None to 0.98115, saving model to ../models/reg_stage2_best.keras

Epoch 1: finished saving model to ../models/reg_stage2_best.keras
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 86s 39ms/step - accuracy: 0.9570 - loss: 0.6265 - val_accuracy: 0.9811 - val_loss: 0.5342 - learning_rate: 1.0000e-04
Epoch 2/12
1589/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9816 - loss: 0.5413
Epoch 2: val_accuracy improved from 0.98115 to 0.98932, saving model to ../models/reg_stage2_best.keras

Epoch 2: finished saving model to ../models/reg_stage2_best.keras
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 41s 25ms/step - accuracy: 0.9816 - loss: 0.5412 - val_accuracy: 0.9893 - val_loss: 0.5061 - learning_rate: 1.0000e-04
Epoch 3/12
1590/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9872 - loss: 0.5175
Epoch 3: val_accuracy improved from 0.98932 to 0.99175, saving model to ../models/reg_stage2_best.keras

Epoch 3: finished saving model to ../models/reg_stage2_best.keras
1592/1592 ━━━

In [3]:
y = pipeline.val_df["label"].values
pv = model.predict(val_ds, verbose=0)          # labels in val_ds are ignored by predict
pt = model.predict(pipeline.make_test_dataset(), verbose=1)
np.save("../models/reg_val_probs.npy", pv); np.save("../models/reg_test_probs.npy", pt)

load = lambda run, f: np.load(f"../{run}/models/{f}")
errs = lambda q: int((q.argmax(axis=1) != y).sum())
p = {"run1_eff": load("backup_run1", "val_probs.npy"), "run1_r50": load("backup_run1", "val_probs_r50.npy"),
     "run2_eff": load("backup_run2", "val_probs.npy"), "run2_r50": load("backup_run2", "val_probs_r50.npy")}

print(f"Regularised EfficientNetB0 errors: {errs(pv)} | accuracy {1 - errs(pv) / len(y):.4f}")
print("Baseline single models:", {k: errs(v) for k, v in p.items()})
print("4-model ensemble:", errs(sum(p.values()) / 4))
print("5-model ensemble (4 + regularised):", errs((sum(p.values()) + pv) / 5))
print("Regularised EfficientNetB0 + run-1 ResNet50V2:", errs((pv + p["run1_r50"]) / 2))

921/921 ━━━━━━━━━━━━━━━━━━━━ 23s 22ms/step
Regularised EfficientNetB0 errors: 60 | accuracy 0.9953
Baseline single models: {'run1_eff': 73, 'run1_r50': 63, 'run2_eff': 75, 'run2_r50': 75}
4-model ensemble: 43
5-model ensemble (4 + regularised): 44
Regularised EfficientNetB0 + run-1 ResNet50V2: 40


In [4]:
tf.keras.utils.set_random_seed(7)
model2 = tf.keras.models.load_model("../models/stage1_best.keras")
base2 = model2.get_layer("efficientnetb0"); base2.trainable = True
for layer in base2.layers:
    layer.trainable = (layer.name.startswith(("block6", "block7", "top"))
                       and not isinstance(layer, layers.BatchNormalization))
model2.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=0.05),
               loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])

cb2 = [tf.keras.callbacks.ModelCheckpoint("../models/reg2_stage2_best.keras", monitor="val_accuracy",
                                          save_best_only=True, verbose=0),
       tf.keras.callbacks.CSVLogger("../models/reg2_stage2_log.csv"),
       tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
       tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=1, min_lr=1e-6, verbose=1)]
model2.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=cb2)
model2.save("../models/reg2_final.keras")

pv2 = model2.predict(val_ds, verbose=0)
np.save("../models/reg2_val_probs.npy", pv2)
np.save("../models/reg2_test_probs.npy", model2.predict(pipeline.make_test_dataset(), verbose=0))
print("Regularised run 2 errors:", errs(pv2), "| run 1:", errs(pv))
print("Both regularised runs averaged:", errs((pv + pv2) / 2))

Epoch 1/12


I0000 00:00:1791787400.281126    1999 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_178489__.254


1590/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9584 - loss: 0.6243

I0000 00:00:1791787442.705991    1998 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_178489__.254


1592/1592 ━━━━━━━━━━━━━━━━━━━━ 69s 33ms/step - accuracy: 0.9584 - loss: 0.6242 - val_accuracy: 0.9865 - val_loss: 0.5236 - learning_rate: 1.0000e-04
Epoch 2/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - accuracy: 0.9829 - loss: 0.5390 - val_accuracy: 0.9870 - val_loss: 0.5057 - learning_rate: 1.0000e-04
Epoch 3/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - accuracy: 0.9875 - loss: 0.5170 - val_accuracy: 0.9895 - val_loss: 0.5003 - learning_rate: 1.0000e-04
Epoch 4/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 41s 26ms/step - accuracy: 0.9903 - loss: 0.5053 - val_accuracy: 0.9921 - val_loss: 0.4919 - learning_rate: 1.0000e-04
Epoch 5/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 41s 25ms/step - accuracy: 0.9925 - loss: 0.4979 - val_accuracy: 0.9918 - val_loss: 0.4904 - learning_rate: 1.0000e-04
Epoch 6/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 41s 25ms/step - accuracy: 0.9932 - loss: 0.4937 - val_accuracy: 0.9912 - val_loss: 0.4897 - learning_rate: 1.0000e-04
Epoch 7/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/

In [5]:
import pandas as pd
y = pipeline.val_df["label"].values
errs = lambda q: int((q.argmax(axis=1) != y).sum())
lv = lambda run, f: np.load(f"../{run}/models/{f}")

v = [lv("backup_run1", "val_probs.npy"), lv("backup_run1", "val_probs_r50.npy"),
     lv("backup_run2", "val_probs.npy"), lv("backup_run2", "val_probs_r50.npy"), pv, pv2]
t = [lv("backup_run1", "test_probs.npy"), lv("backup_run1", "test_probs_r50.npy"),
     lv("backup_run2", "test_probs.npy"), lv("backup_run2", "test_probs_r50.npy"),
     np.load("../models/reg_test_probs.npy"), np.load("../models/reg2_test_probs.npy")]

print("Baseline EfficientNetB0 pair (run 1 + rerun):", errs((v[0] + v[2]) / 2))
print("Regularised EfficientNetB0 pair:", errs((v[4] + v[5]) / 2))
e6 = sum(v) / 6
print(f"6-model equal-weight ensemble: {errs(e6)} errors | accuracy {1 - errs(e6) / len(y):.4f}")

t6 = sum(t) / 6
sub = pd.DataFrame({"image_id": pipeline.test_df["image_id"].values,
                    "class": [pipeline.idx_to_class[i] for i in t6.argmax(axis=1)]})
sub.to_csv("../submissions/submission_v4_ensemble6.csv", index=False)
sample = pd.read_csv("../data/sample_submission.csv"); v3 = pd.read_csv("../submissions/submission_v3_ensemble4.csv")
print("Shape:", sub.shape, "| columns match:", list(sub.columns) == list(sample.columns),
      "| IDs match:", (sub["image_id"].values == sample["image_id"].values).all())
print(f"Predictions changed vs v3: {(sub['class'].values != v3['class'].values).sum()} of {len(sub)}")

Baseline EfficientNetB0 pair (run 1 + rerun): 66
Regularised EfficientNetB0 pair: 57
6-model equal-weight ensemble: 45 errors | accuracy 0.9965
Shape: (29443, 2) | columns match: True | IDs match: True
Predictions changed vs v3: 106 of 29443


In [6]:
path = "../docs/experiment_log.md"
lines = open(path).read().split("\n")
last_row = max(k for k, l in enumerate(lines) if l.startswith("| "))
rows = [
 "| 11 | EfficientNetB0 stage 2 with AdamW (weight decay 0.05) + label smoothing 0.1, seed 42 (from the stage-1 checkpoint of #1) | 0.9953 | - | 60 | not submitted |",
 "| 12 | Same as #11, seed 7 | 0.9953 | - | 60 | not submitted |",
 "| 13 | Average of #11 and #12 (baseline pair #2 + #7: 66 errors) | 0.9955 | - | 57 | not submitted |",
 "| 14 | Equal-weight average of 6 models (#2, #4, #7, #8, #11, #12) | 0.9965 | - | 45 | not submitted (rule fixed in advance: submit only if below 43) |",
]
for i, r in enumerate(rows): lines.insert(last_row + 1 + i, r)
lines.append("Rows 11-14: regularisation experiment. Regularised runs repeated at 60 errors versus 73 and 75 for the baselines; the 6-model ensemble did not beat the 43-error 4-model ensemble.")
open(path, "w").write("\n".join(lines))
print(open(path).read())

# Experiment log (ENGE707 Kaggle)

| # | Model / setup | Val acc | Val loss | Val errors | Kaggle public |
|---|---|---|---|---|---|
| 1 | EfficientNetB0 stage 1 (frozen, lr 1e-3, 5 epochs) | 0.9457 | 0.1517 | - | - |
| 2 | EfficientNetB0 stage 2 (top blocks unfrozen, lr 1e-4, ReduceLROnPlateau, 12 epochs) | 0.9943 | 0.0206 | 73 | 0.9767 (v1) |
| 3 | ResNet50V2 stage 1 (frozen, lr 1e-3, 5 epochs) | 0.9687 | 0.0966 | - | - |
| 4 | ResNet50V2 stage 2 (conv5 unfrozen, lr 1e-4, early stopped epoch 11, restored epoch 8) | 0.9951 | 0.0189 | 63 | not submitted |
| 5 | EfficientNetB0 + horizontal-flip TTA | 0.9943 | - | 73 | not submitted |
| 6 | Ensemble: average of #2 and #4 probabilities | 0.9964 | 0.0138 (log loss) | 46 | 0.9786 (v2) |
| 7 | Rerun of #2 (EfficientNetB0, identical code, seed 42 set) | 0.9941 | - | 75 | not submitted |
| 8 | Rerun of #4 (ResNet50V2, identical code, seed 42 set) | 0.9941 | - | 75 | not submitted |
| 9 | Rerun of #6 (ensemble of #7 and #8) | 0.9958 | - | 53 | 